# Stage 1: Preprocessing & Phase Spectrum Extraction
**Project:** KAN-Driven Phase-Spectrum Analysis for Deepfake Detection  
**Environment:** Kaggle / Local

In [ ]:
!pip install kagglehub -q

In [ ]:
import numpy as np, cv2, os, glob
import matplotlib.pyplot as plt
from typing import Dict, Tuple, Optional
import kagglehub

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# --- Guaranteed path detection ---
INPUT_DIR = kagglehub.dataset_download('awsaf49/artifact-dataset')
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(INPUT_DIR, '..')

print(f'Dataset downloaded to: {INPUT_DIR}')
print(f'Contents:')
for item in sorted(os.listdir(INPUT_DIR))[:15]:
    f = os.path.join(INPUT_DIR, item)
    print(f'  {"[DIR]" if os.path.isdir(f) else "[FILE]"} {item}')

In [ ]:
# Cell 2: PhaseSpectrumExtractor

class PhaseSpectrumExtractor:
    _LUMA_R, _LUMA_G, _LUMA_B = 0.2989, 0.5870, 0.1140
    def __init__(self, target_size=(256, 256)):
        self.target_size = target_size
    def load_image(self, path):
        bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if bgr is None: raise ValueError(f'Cannot read: {path}')
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    def rgb_to_grayscale(self, rgb):
        f = rgb.astype(np.float64)
        return self._LUMA_R*f[:,:,0] + self._LUMA_G*f[:,:,1] + self._LUMA_B*f[:,:,2]
    def process(self, image_path):
        rgb = self.load_image(image_path)
        gray = self.rgb_to_grayscale(rgb)
        gray_r = cv2.resize(gray, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_CUBIC)
        fft_shifted = np.fft.fftshift(np.fft.fft2(gray_r))
        phase = np.angle(fft_shifted)
        p_min, p_max = phase.min(), phase.max()
        phase_norm = (phase - p_min) / (p_max - p_min) if p_max > p_min else np.zeros_like(phase)
        mag_log = np.log1p(np.abs(fft_shifted))
        return {'original_rgb': rgb, 'grayscale_resized': gray_r,
                'magnitude_log': mag_log, 'phase_normalized': phase_norm}

print('PhaseSpectrumExtractor defined.')

In [ ]:
# Cell 3: Find Image & Run
sample_image = None
for ext in ['**/*.jpg', '**/*.png', '**/*.jpeg', '**/*.bmp', '**/*.webp']:
    found = glob.glob(os.path.join(INPUT_DIR, ext), recursive=True)
    if found:
        sample_image = found[0]
        print(f'Found {len(found)} images with pattern {ext}')
        break

if sample_image is None:
    print(f'No images found! Directory tree (first 30):')
    count = 0
    for root, dirs, files in os.walk(INPUT_DIR):
        for f in files:
            print(f'  {os.path.join(root, f)}')
            count += 1
            if count >= 30: break
        if count >= 30: break
else:
    print(f'Using: {sample_image}')
    extractor = PhaseSpectrumExtractor()
    results = extractor.process(sample_image)

    assert results['phase_normalized'].shape == (256, 256)
    assert 0.0 <= results['phase_normalized'].min()
    assert results['phase_normalized'].max() <= 1.0
    print(f'Phase map: {results["phase_normalized"].shape}, range [0,1] ✓')

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle('Phase Spectrum Extraction Pipeline', fontsize=14, fontweight='bold')
    axes[0].imshow(results['original_rgb']); axes[0].set_title('(a) RGB'); axes[0].axis('off')
    axes[1].imshow(results['grayscale_resized'], cmap='gray'); axes[1].set_title('(b) Gray 256×256'); axes[1].axis('off')
    axes[2].imshow(results['magnitude_log'], cmap='inferno'); axes[2].set_title('(c) FFT Magnitude'); axes[2].axis('off')
    axes[3].imshow(results['phase_normalized'], cmap='twilight', vmin=0, vmax=1); axes[3].set_title('(d) Phase [0,1]'); axes[3].axis('off')
    plt.tight_layout(); plt.show()